# Despliegue del modelo en un endpoint productivo

Este notebook documenta los pasos para empaquetar el modelo seleccionado, exponerlo vía API (FastAPI) y preparar los artefactos necesarios para una imagen Docker, siguiendo la rúbrica del proyecto final.


## Objetivos
- Cargar el modelo y pipeline entrenados.
- Definir la función de inferencia y validar un ejemplo de predicción.
- Proporcionar un esqueleto de servicio FastAPI listo para producción.
- Documentar los archivos necesarios para construir la imagen Docker.
- Especificar pruebas básicas y recomendaciones post-despliegue.


In [11]:
# Importación de librerías y artefactos
import json
from pathlib import Path
from typing import Dict, List

import joblib
import numpy as np
import pandas as pd


def _find_project_root(start_path: Path) -> Path:
    current = start_path if start_path.is_dir() else start_path.parent
    for candidate in [current, *current.parents]:
        if (candidate / "config.json").exists():
            return candidate
    raise FileNotFoundError(
        "No se encontró config.json en la jerarquía de directorios. Ejecuta el notebook desde el proyecto."  # noqa: E501
    )


START_PATH = Path(__file__).resolve() if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = _find_project_root(START_PATH)
OUTPUTS_DIR = PROJECT_ROOT / "mlops_pipeline" / "outputs"
MODELS_DIR = PROJECT_ROOT / "mlops_pipeline" / "models"
CONFIG_PATH = PROJECT_ROOT / "config.json"

with CONFIG_PATH.open("r", encoding="utf-8") as fp:
    config = json.load(fp)

model_metadata_path = MODELS_DIR / "model_metadata.json"
with model_metadata_path.open("r", encoding="utf-8") as fp:
    metadata = json.load(fp)

model_path = MODELS_DIR / f"best_model_{metadata['model_name']}.joblib"
feature_pipeline_path = Path(metadata["feature_pipeline_path"])

model = joblib.load(model_path)
feature_pipeline = joblib.load(feature_pipeline_path)

print(f"Modelo cargado: {metadata['model_name']}")
print(f"Total de features: {metadata['feature_count']}")


Modelo cargado: logistic_regression
Total de features: 179


In [12]:
# Obtención de un registro de ejemplo para validar la inferencia
data_path = PROJECT_ROOT / config["pipeline_config"]["data_path"]
raw_df = pd.read_csv(data_path, sep=";", encoding="utf-8")
raw_df.columns = (
    raw_df.columns
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace("\t", "", regex=False)
)

sample_record = raw_df.drop(columns=["Target"]).iloc[0].to_dict()
sample_record


{'Marital status': 1.0,
 'Application mode': 17.0,
 'Application order': 5.0,
 'Course': 171.0,
 'Daytime/evening attendance': 1.0,
 'Previous qualification': 1.0,
 'Previous qualification (grade)': 122.0,
 'Nacionality': 1.0,
 "Mother's qualification": 19.0,
 "Father's qualification": 12.0,
 "Mother's occupation": 5.0,
 "Father's occupation": 9.0,
 'Admission grade': 127.3,
 'Displaced': 1.0,
 'Educational special needs': 0.0,
 'Debtor': 0.0,
 'Tuition fees up to date': 1.0,
 'Gender': 1.0,
 'Scholarship holder': 0.0,
 'Age at enrollment': 20.0,
 'International': 0.0,
 'Curricular units 1st sem (credited)': 0.0,
 'Curricular units 1st sem (enrolled)': 0.0,
 'Curricular units 1st sem (evaluations)': 0.0,
 'Curricular units 1st sem (approved)': 0.0,
 'Curricular units 1st sem (grade)': 0.0,
 'Curricular units 1st sem (without evaluations)': 0.0,
 'Curricular units 2nd sem (credited)': 0.0,
 'Curricular units 2nd sem (enrolled)': 0.0,
 'Curricular units 2nd sem (evaluations)': 0.0,
 'Cur

In [13]:
# Función de inferencia reutilizable
def predict_dropout(payload: Dict) -> Dict:
    """Recibe un diccionario con las variables del estudiante y retorna probabilidades y clase."""
    input_df = pd.DataFrame([payload])
    transformed = feature_pipeline.transform(input_df)
    proba = model.predict_proba(transformed)[0]
    classes = model.classes_
    prediction = classes[np.argmax(proba)]

    return {
        "prediction": prediction,
        "probabilities": {cls: float(p) for cls, p in zip(classes, proba)},
    }

predict_dropout(sample_record)


{'prediction': 'Dropout',
 'probabilities': {'Dropout': 0.5667277602352052,
  'Enrolled': 0.18077115390205403,
  'Graduate': 0.2525010858627408}}

## Esqueleto del servicio FastAPI
El siguiente archivo `app/main.py` expone el endpoint `/predict` con validación de esquema y manejo de lotes. Puede generarse automáticamente ejecutando la celda de abajo (creará la carpeta `app/`).


In [14]:
# Preparar directorio de la aplicación FastAPI
APP_DIR = PROJECT_ROOT / "app"
APP_DIR.mkdir(parents=True, exist_ok=True)
app_main_path = APP_DIR / "main.py"



In [15]:
# Escritura del servicio FastAPI en disco
from textwrap import dedent

service_code = dedent('''
"""Servicio de inferencia FastAPI para el modelo de deserción estudiantil."""
from pathlib import Path
from typing import Dict, List

import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

APP_ROOT = Path(__file__).resolve().parent
MODELS_DIR = APP_ROOT.parent / "mlops_pipeline" / "models"
OUTPUTS_DIR = APP_ROOT.parent / "mlops_pipeline" / "outputs"

metadata_path = MODELS_DIR / "model_metadata.json"
if metadata_path.suffix == ".joblib":
    metadata = joblib.load(metadata_path)
else:
    import json

    with metadata_path.open("r", encoding="utf-8") as fp:
        metadata = json.load(fp)

model = joblib.load(MODELS_DIR / f"best_model_{metadata['model_name']}.joblib")
feature_pipeline = joblib.load(OUTPUTS_DIR / "feature_pipeline.joblib")

app = FastAPI(title="Student Dropout Predictor", version="1.0.0")


class StudentRequest(BaseModel):
    records: List[Dict]


class BatchPrediction(BaseModel):
    predictions: List[str]
    probabilities: List[Dict[str, float]]


@app.get("/health", tags=["infra"])
async def health_check() -> Dict[str, str]:
    return {"status": "ok"}


@app.post("/predict", response_model=BatchPrediction, tags=["model"])
async def predict(request: StudentRequest) -> BatchPrediction:
    input_df = pd.DataFrame(request.records)
    transformed = feature_pipeline.transform(input_df)
    probabilities = model.predict_proba(transformed)
    classes = model.classes_
    preds = model.predict(transformed)

    return BatchPrediction(
        predictions=preds.tolist(),
        probabilities=[{cls: float(prob) for cls, prob in zip(classes, row)} for row in probabilities],
    )
''')

with app_main_path.open("w", encoding="utf-8") as f:
    f.write(service_code)

print(f"Servicio FastAPI escrito en: {app_main_path}")


Servicio FastAPI escrito en: c:\Users\jesus\OneDrive\Documentos\Proyecto Final ML\Proyecto\app\main.py


### Ejecución local del servicio
```bash
python -m uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload
```

Prueba rápida (usando `httpie` o `curl`):
```bash
http POST http://127.0.0.1:8000/predict \
  records:='[{"Marital status":1, "Application mode":17, "Application order":5, ...}]'
```

> El payload debe contener exactamente las columnas del dataset (puedes reutilizar `sample_record`).


## Dockerfile sugerido
La siguiente celda genera un `Dockerfile` compatible con el despliegue en contenedores y una configuración mínima de `.dockerignore`. Ajusta la imagen base según las políticas de tu organización.


In [16]:
%%writefile Dockerfile
FROM python:3.13-slim

WORKDIR /app

# Copiamos archivos de configuración y requirements
COPY requirements.txt ./
RUN pip install --no-cache-dir -r requirements.txt

# Copiamos el código fuente
COPY mlops_pipeline ./mlops_pipeline
COPY app ./app

ENV PYTHONPATH="/app/mlops_pipeline/src"

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]


Writing Dockerfile


In [17]:
%%writefile .dockerignore
__pycache__/
.ipynb_checkpoints/
.git/
mlops_pipeline/outputs/
mlops_pipeline/models/*.joblib
mlops_pipeline/models/*.pkl



Writing .dockerignore


### Pruebas recomendadas antes del despliegue
1. **Unitarias:** validar la función `predict_dropout` con casos esperados y edge-cases (valores nulos, categorías fuera de catálogo).
2. **Contract tests:** verificar que el esquema de entrada de la API no cambia usando `pydantic` y pruebas de integración.
3. **Smoke test:** levantar el contenedor y consumir el endpoint `/health` y `/predict` con el payload de muestra.
4. **SonarCloud:** ejecutar análisis estático sobre la carpeta `app/` y el código de `mlops_pipeline/src`.


## Próximos pasos
- Publicar la imagen en el registro corporativo y definir pipelines de despliegue (Jenkins/GitHub Actions).
- Configurar observabilidad (logs estructurados, métricas, trazas) y alarmas en base a la respuesta de la API.
- Conectar el servicio con el módulo de monitoreo para registrar cada lote predicho y facilitar la detección de drift.
- Documentar el flujo end-to-end y agregarlo al `README.md` del repositorio.


### Pruebas recomendadas antes del despliegue
1. **Unitarias:** validar la función `predict_dropout` con casos esperados y edge-cases (valores nulos, categorías fuera de catálogo).
2. **Contract tests:** verificar que el esquema de entrada de la API no cambia usando `pydantic` y pruebas de integración.
3. **Smoke test:** levantar el contenedor y consumir el endpoint `/health` y `/predict` con el payload de muestra.
4. **SonarCloud:** ejecutar análisis estático sobre la carpeta `app/` y el código de `mlops_pipeline/src`.
